# 92장 · 기준선 통과율과 오류 분석

평가 결과는 같은 판본과 같은 기준의 기록끼리 집계해야 합니다. 실패한 문항은 기대 답변과 실제 출력을 함께 읽습니다.

91장의 최소 구현을 고정 질문 일곱 개로 평가합니다. 답변 출력 여부와 수용 기준 통과 여부를 구분하고, 실패한 항목을 다음 수정의 기준선으로 남깁니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 팀의 실행 행으로 상태와 위치 오류 세기

앞의 7건 예시는 당시 팀의 개발 기록입니다. 이번 묶음에서는 91장에서 실행한 행을 분모로 삼고, 상태와 문서 및 쪽수가 모두 맞는 행을 다시 셉니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter91.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter92-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter91.json"
sample_file = here / "chapter92-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['runs'])

교재 예시 chapter92-start.json
[{'id': 'Q01', 'question': '장비 A의 점검 전 확인 사항은?', 'expected_status': 'answered', 'expected_doc': 'RULE-A', 'expected_page': 1, 'actual_status': 'answered', 'actual_doc': 'RULE-A', 'actual_page': None, 'http_status': 200}, {'id': 'Q02', 'question': '점검 전 확인 사항은?', 'expected_status': 'need_clarification', 'expected_doc': None, 'expected_page': None, 'actual_status': 'need_clarification', 'actual_doc': None, 'actual_page': None, 'http_status': 200}, {'id': 'Q03', 'question': '장비 A를 다음 달에 교체하나요?', 'expected_status': 'no_evidence', 'expected_doc': None, 'expected_page': None, 'actual_status': 'answered', 'actual_doc': 'RULE-A', 'actual_page': None, 'http_status': 200}]


두 번째 줄은 읽은 파일의 `runs` 값입니다.

## 같은 판본의 통과 건수 집계

기준선 카드　프로젝트 PROJECT-EQUIPMENT-01 / 문서 rules-v1 / 질문 eval-v1 / 로컬 규칙 thin / 일곱 문항 중 두 문항 통과. 여기에 모델이 개입한 적은 없습니다. 다음 변경을 비교할 때는 질문·원문을 그대로 쓰고 무엇을 바꿨는지 한 줄로 남깁니다.


### 셀 01 · 판본별 통과 건수 집계

baseline.json에서 질문 번호와 통과 여부만 꺼낸 자료입니다. 빈칸을 잘된 것으로 세지 않도록 실제 True인 행만 셉니다.


In [1]:
records = [{'id': 'Q01', 'passed': False},
 {'id': 'Q02', 'passed': True},
 {'id': 'Q03', 'passed': False},
 {'id': 'Q04', 'passed': False},
 {'id': 'Q05', 'passed': False},
 {'id': 'Q06', 'passed': True},
 {'id': 'Q07', 'passed': False}]
print(sum(row["passed"] is True for row in records), "/", len(records))

2 / 7


2 / 7이 나옵니다. 같은 일곱 질문을 보며 코드를 수정하므로 이 비율은 공개 개발 사례의 결과입니다. 다른 사용자 질문에 대한 정확도로 발표할 수 없습니다.

2/7을 발표 자료에 옮길 때는 어떤 판본으로 어떤 일곱 문항을 실행했는지 함께 표시합니다. 다음 판에서 질문을 빼거나 추가하면 통과 비율이 높아져도 같은 조건의 개선인지 알기 어렵습니다. 실패 사유까지 남겨야 고칠 기능을 선택할 수 있습니다.


#### 통과율의 분모를 고정하는 이유

<mark>다른 판본이나 다른 문항이 섞이면 수정 효과를 분리할 수 없습니다.</mark> 2/7이라는 숫자 옆에 일곱 문항의 목록과 실패 필드를 남기세요. 다음 실행에서 실패 하나가 없어졌을 때 같은 문항을 고친 결과인지 확인할 수 있습니다.

선수 개념 · 22장 정답 문단의 검색 순위 평가

#### 분자와 분모로 다시 읽기

$$\frac{2}{7}\times100\%\approx28.6\%$$

같은 판본의 일곱 문항 중 두 문항이 통과한 기준선입니다.

### 2/7을 실제 질문 번호로 풀어 읽기

passed가 True인 두 행은 Q02와 Q06입니다. 다음 판본도 2/7이라면 이 두 ID가 여전히 통과하는지 확인하세요. 예를 들어 Q01이 새로 통과하고 Q02가 실패하면 합계가 그대로여서 어느 질문이 달라졌는지 드러나지 않습니다.


## 기대값과 실제 답변 비교

실패 기록 예시　Q03 / 기대 no_evidence, 인용 없음 / 실제 answered, RULE-A 인용 / 원인 장비명만 일치하면 점검 문장을 반환 / 수정 후보 일정 문의를 답변 불가로 처리. 반면 Q01은 답의 의미가 맞아도 출처에 절과 개정일이 빠졌습니다. 서로 다른 두 문제를 ‘답변 품질 개선’ 한 줄에 뭉치면 수정 순서를 정하기 어렵습니다.


### 팀 참고 예시 · 실패한 평가 조건


In [2]:
checks = {'status': True,
 'source_ids': True,
 'required_facts': True,
 'source_metadata': False,
 'allowed_sources': True,
 'execution_mode': True}
print([name for name, passed in checks.items() if not passed])

['source_metadata']


source_metadata 한 항목이 남습니다. 출처 ID는 이미 맞았으므로 새 검색기를 도입하기에 앞서 응답에 담는 필드를 확인합니다.


## 같은 기준선에서 첫 수정 하나를 고르는 회의

일곱 공개 개발 문항 중 두 개가 통과했습니다. 다음 점검 때 2/7이라는 숫자만 보고 무엇을 고쳤는지 짐작하지 않도록, 실패 두 건의 중간 결과를 따로 보관합니다.

### Q01과 Q03의 결함 분류표

<table><thead><tr><th scope="col">문항과 실제 관찰</th><th scope="col">어느 지점에서 기대와 갈렸나</th><th scope="col">수정 후 같은 입력에서 볼 값</th></tr></thead><tbody><tr><td>Q01 · 답에 전원 표시등, 출처는 RULE-A의 ID뿐</td><td>자료 선택 뒤 출처를 만드는 단계</td><td>CR-02의 여섯 필드가 원문과 일치</td></tr><tr><td>Q03 · 교체 일정을 물었는데 점검 안내와 RULE-A 반환</td><td>질문의 목적을 검사하기 전 장비명만 보고 조회</td><td>no_evidence와 빈 출처</td></tr></tbody></table>

팀은 Q03부터 수정하기로 했습니다. 근거가 없는 질문에 단정적인 답을 내는 문제가 사용자의 다음 행동을 바꿀 수 있기 때문입니다. Q01은 별도의 출처 수정 항목으로 남깁니다.

### 셀 04 · 부분 수정 시 전체 점수가 얼마나 변할지 검산

Q03 하나만 고친 경우를 가정합니다. 원래 records를 수정하지 않고 새 목록에서 바뀐 문항 ID를 찾아 보세요.

In [3]:
projected = [{**row, "passed": True} if row["id"] == "Q03" else row.copy()
             for row in records]
changed_ids = [old["id"] for old, new in zip(records, projected)
               if old["passed"] != new["passed"]]
print("가정한 변화", changed_ids, sum(row["passed"] for row in projected), "/", len(projected))

가정한 변화 ['Q03'] 3 / 7


3/7은 Q03 하나만 통과한다고 가정한 숫자입니다. 이 값을 개선 결과 칸에 옮기기 전, 수정한 서비스를 일곱 입력으로 다시 실행해야 합니다.

<mark>우선순위 표에는 점수와 함께 최초로 어긋난 단계를 적습니다.</mark> Q01과 Q03은 같은 실패 1건씩이지만 출처 조립을 고쳐도 Q03의 질문 목적 오류는 남습니다.

93장 상태 설계에는 교체 일정 문의가 검색을 거치지 않고 끝나는 경로를 추가합니다.

## 팀 사례에 적용하기 · 팀의 실행 행으로 상태와 위치 오류 세기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
issues = []
for row in incoming["runs"]:
    bad = [field for field in ("status", "doc", "page")
           if row["expected_" + field] != row["actual_" + field]]
    if bad:
        issues.append({"id": row["id"], "question": row["question"], "mismatch": bad})
new = {"quality_reference": incoming["quality_reference"], "issues": issues, "total": len(incoming["runs"]), "documents": incoming["documents"]}
print("통과 / 전체", new["total"] - len(issues), "/", new["total"])

통과 / 전체 1 / 3


통과한 행과 전체 행이 같은 파일에서 계산됩니다. 실패한 행에는 문제가 생긴 필드 이름이 남으므로 93장에서 수정할 분기를 고를 수 있습니다.

In [ ]:
output_file = project_root / "team-project/final/chapter92.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter92.json


93장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

개발용 eval-v1에서 2/7이었던 기준선, Q01의 출처 위치 누락, Q03의 질문 의도 오류를 93장 설계 회의에 넘기세요. 모델 호출은 0회였고 운영 비용과 지연은 측정하지 않았습니다.
